In [8]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

SOURCE_PATH = DATA_DIR / "DSL-StrongPasswordData.csv"
OUTPUT_PATH = DATA_DIR / "keystroke_raw.parquet"

print("📥 Loading DSL dataset...")
df = pd.read_csv(SOURCE_PATH)
print(f"Loaded {len(df)} rows.")

# --- Normalize IDs ---
df.rename(columns={"subject": "user_id", "sessionIndex": "session_id"}, inplace=True)

def normalize_user(u):
    u = str(u)
    if u.startswith("s"):
        return int(u.lstrip("s").lstrip("0") or 0)
    return int(u)

df["user_id"] = df["user_id"].apply(normalize_user)
df["session_id"] = df["session_id"].astype(int)

# --- Identify timing feature columns ---
exclude_cols = {"user_id", "session_id", "rep"}
feature_cols = [c for c in df.columns if c not in exclude_cols]

print(f"Found {len(feature_cols)} timing feature columns:")
print(feature_cols)

# --- Add timestamp (rep index as proxy) ---
df["timestamp"] = df["rep"].astype(float)

# --- Make sure timing columns are numeric ---
df[feature_cols] = df[feature_cols].astype("float32")

# --- Final output: NO 'features' object column ---
df_out = df[["user_id", "session_id", "timestamp"] + feature_cols].copy()

print("\nSample row:")
print(df_out.iloc[0])

# --- Save with fastparquet ---
print(f"\n💾 Saving cleaned data → {OUTPUT_PATH}")
df_out.to_parquet(OUTPUT_PATH, index=False, engine="fastparquet")
print("✅ Done! Keystroke dataset is ready for feature extraction.")

📥 Loading DSL dataset...
Loaded 20400 rows.
Found 31 timing feature columns:
['H.period', 'DD.period.t', 'UD.period.t', 'H.t', 'DD.t.i', 'UD.t.i', 'H.i', 'DD.i.e', 'UD.i.e', 'H.e', 'DD.e.five', 'UD.e.five', 'H.five', 'DD.five.Shift.r', 'UD.five.Shift.r', 'H.Shift.r', 'DD.Shift.r.o', 'UD.Shift.r.o', 'H.o', 'DD.o.a', 'UD.o.a', 'H.a', 'DD.a.n', 'UD.a.n', 'H.n', 'DD.n.l', 'UD.n.l', 'H.l', 'DD.l.Return', 'UD.l.Return', 'H.Return']

Sample row:
user_id            2.0000
session_id         1.0000
timestamp          1.0000
H.period           0.1491
DD.period.t        0.3979
UD.period.t        0.2488
H.t                0.1069
DD.t.i             0.1674
UD.t.i             0.0605
H.i                0.1169
DD.i.e             0.2212
UD.i.e             0.1043
H.e                0.1417
DD.e.five          1.1885
UD.e.five          1.0468
H.five             0.1146
DD.five.Shift.r    1.6055
UD.five.Shift.r    1.4909
H.Shift.r          0.1067
DD.Shift.r.o       0.7590
UD.Shift.r.o       0.6523
H.o        